In [1]:
import pickle
import numpy as np
from scipy.io import loadmat
import pandas as pd
import os
import gc

In [2]:
neuron_psth = np.load("/media/ubuntu/sda/TrippleN/psth/rmi_GoodUnit_240629_JianJian_NSD1000_LOC_g2.npy")
processed = loadmat("/media/ubuntu/sda/TrippleN/Processed/Processed_ses01_240629_M1_2.mat")

columns = ['B_SI', 'F_SI', 'O_SI', 'UnitType', 'best_r_time1', 'best_r_time2', 'pos', 'reliability_basic', 'reliability_best', 'reliability_find_testset', 'snr', 'snrmax']

In [7]:
a = loadmat("/media/ubuntu/sda/TrippleN/GoodUnit/GoodUnit_250916_MaoDan_NSD1000_LOC_g6.mat")

NotImplementedError: Please use HDF reader for matlab v7.3 files, e.g. h5py

In [3]:
processed_dict = {}
for col in columns:
    if col in processed:
        data = processed[col]
        if data.ndim == 2 and data.shape[0] == 1:
            processed_dict[col] = data.flatten()
        else:
            processed_dict[col] = data.flatten() if data.ndim > 1 else data

processed_df = pd.DataFrame(processed_dict)

In [9]:
processed_df

,B_SI,F_SI,O_SI,UnitType,best_r_time1,best_r_time2,pos,reliability_basic,reliability_best,reliability_find_testset,snr,snrmax
0,-0.421162,0.246799,0.086835,1,100,130,25.727272,-0.023474,-0.011826,-0.046935,0.078273,11.336235
1,-0.096438,0.137831,-0.041969,2,70,250,37.957912,0.445708,0.372180,0.431578,0.288934,16.409863
2,-0.333023,0.270188,0.067780,2,70,290,35.751434,0.598082,0.648545,0.641072,0.316985,12.663879
3,-0.065084,-0.622890,0.773016,3,140,250,34.084335,0.020941,0.113260,0.002799,0.355949,10.878104
4,-0.129649,0.131297,-0.014097,3,90,100,37.218349,-0.060751,0.063663,-0.025250,0.324834,16.761240
...,...,...,...,...,...,...,...,...,...,...,...,...
447,-0.273154,1.642164,-1.334561,3,90,340,3764.433594,0.729052,0.709151,0.758312,0.245467,22.207035
448,0.733266,-1.348100,0.304694,2,110,350,3768.048828,0.567840,0.652751,0.639798,0.407426,11.393444
449,0.918947,-0.761808,-0.302395,3,60,370,3787.379639,0.689095,0.638572,0.721381,0.415798,17.896883
450,0.193084,-1.113594,0.786816,2,140,320,3795.444092,0.602624,0.739421,0.761881,0.319711,9.338350


In [8]:
processed

{'__header__': b'MATLAB 5.0 MAT-file, Platform: PCWIN64, Created on: Wed Nov 26 13:16:46 2025',
 '__version__': '1.0',
 '__globals__': [],
 'B_SI': array([[-4.21161950e-01, -9.64382291e-02, -3.33022922e-01,
         -6.50835559e-02, -1.29648685e-01,  3.46119881e-01,
          6.39732927e-02, -9.70714986e-02, -1.25734851e-01,
          1.76066786e-01, -1.52522951e-01, -3.70583743e-01,
          5.99421620e-01, -7.20049977e-01, -1.71259463e-01,
          1.43642053e-01, -4.10547197e-01,  2.67693162e-01,
         -3.06675822e-01,  2.06577748e-01, -5.14709473e-01,
          8.79273772e-01, -4.74782474e-02, -1.27328885e+00,
         -4.04524207e-02, -6.79991007e-01, -8.45235407e-01,
         -3.67127180e-01, -7.80093074e-01,  5.68940043e-01,
          1.39316395e-01, -1.76768959e-01, -5.84194839e-01,
         -1.22823492e-01, -4.61225510e-01, -4.04468715e-01,
          5.85352302e-01,  5.69532394e-01,  6.52707696e-01,
         -3.45127750e-03, -8.55211616e-01,  2.32177060e-02,
         -6.2

In [11]:
neuron_psth.shape

(452, 1072, 450)

In [ ]:
neuron_psth = neuron_psth[np.where(processed['reliability_basic'][0, :] > 0.4)[0], :, :]
processed_df = processed_df[processed_df['reliability_basic'] > 0.4]

In [ ]:
goodunit_list = sorted(os.listdir('/media/ubuntu/sda/TrippleN/GoodUnit'))
processed_list = sorted(os.listdir("/media/ubuntu/sda/TrippleN/Processed"))

In [12]:
monkey_names = ['JianJian', 'FaCai', 'TuTu', 'MaoDan', 'ZhuangZhuang']
columns = ['B_SI', 'F_SI', 'O_SI', 'UnitType', 'best_r_time1', 'best_r_time2', 'pos', 'reliability_basic', 'reliability_best', 'reliability_find_testset', 'snr', 'snrmax']
psth_dir = "/media/ubuntu/sda/TrippleN/psth"
processed_dir = "/media/ubuntu/sda/TrippleN/Processed"

psth_files = sorted(os.listdir(psth_dir))
processed_files = sorted(os.listdir(processed_dir))

output_dir = "/media/ubuntu/sda/TrippleN/GoodUnit_by_monkey"
os.makedirs(output_dir, exist_ok=True)

print("="*50)
print("逐只猴子处理数据（内存优化版）...")
print("="*50)

for monkey in monkey_names:
    print(f"\n处理猴子: {monkey}")
    print("-"*30)
    
    # 找出属于当前猴子的session
    monkey_psth_files = []
    monkey_processed_files = []
    
    for i, psth_file in enumerate(psth_files):
        parts = psth_file.replace('.npy', '').split('_')
        if len(parts) >= 4 and parts[3] == monkey:
            monkey_psth_files.append(psth_file)
            monkey_processed_files.append(processed_files[i])
    
    if len(monkey_psth_files) == 0:
        print(f"  没有找到 {monkey} 的数据")
        continue
    
    print(f"  找到 {len(monkey_psth_files)} 个sessions")
    
    # 第一步：先统计总神经元数，确定数组大小
    session_neurons = []
    for psth_file in monkey_psth_files:
        # 使用memmap快速获取形状，不加载数据
        with np.load(os.path.join(psth_dir, psth_file), mmap_mode='r') as data:
            n_neurons = data.shape[0]
            session_neurons.append(n_neurons)
    
    total_neurons = sum(session_neurons)
    print(f"  总神经元数: {total_neurons}")
    
    # 获取单个session的形状（用于确定时间维度）
    with np.load(os.path.join(psth_dir, monkey_psth_files[0]), mmap_mode='r') as data:
        n_stimuli = data.shape[1] if data.ndim >= 2 else 1
        n_time = data.shape[2] if data.ndim >= 3 else (data.shape[1] if data.ndim == 2 else 1)
    
    print(f"  PSTH形状预分配: ({total_neurons}, {n_stimuli}, {n_time})")
    print(f"  预计内存占用: {total_neurons * n_stimuli * n_time * 4 / 1024**2:.1f} MB (float32)")
    
    # 第二步：预分配float32数组
    monkey_psth = np.zeros((total_neurons, n_stimuli, n_time), dtype=np.float32)
    monkey_df_list = []
    
    # 第三步：逐个session处理，填入预分配数组
    current_idx = 0
    for j, (psth_file, proc_file) in enumerate(zip(monkey_psth_files, monkey_processed_files)):
        print(f"    Session {j+1}/{len(monkey_psth_files)}: {psth_file} ({session_neurons[j]} neurons)")
        
        # 使用memmap只读加载，不复制到内存
        with np.load(os.path.join(psth_dir, psth_file), mmap_mode='r') as data:
            n_neurons = data.shape[0]
            # 复制到float32数组（只复制需要的部分）
            monkey_psth[current_idx:current_idx+n_neurons] = data.astype(np.float32)
        
        proc_data = loadmat(os.path.join(processed_dir, proc_file))
        
        session_dict = {}
        for col in columns:
            if col in proc_data:
                data = proc_data[col]
                if data.ndim == 2 and data.shape[0] == 1:
                    session_dict[col] = data.flatten()
                else:
                    session_dict[col] = data.flatten() if data.ndim > 1 else data
        
        session_df = pd.DataFrame(session_dict)
        session_df['session_file'] = psth_file.replace('.npy', '')
        session_df['monkey'] = monkey
        
        # 维度检查
        n_df = len(session_df)
        if n_neurons != n_df:
            min_n = min(n_neurons, n_df)
            monkey_psth[current_idx:current_idx+min_n] = monkey_psth[current_idx:current_idx+n_neurons][:min_n]
            session_df = session_df.iloc[:min_n]
            n_neurons = min_n
        
        monkey_df_list.append(session_df)
        current_idx += n_neurons
        
        del proc_data, session_dict, session_df
        gc.collect()
    
    # 合并DataFrame
    print(f"  合并DataFrame...")
    monkey_df = pd.concat(monkey_df_list, ignore_index=True)
    del monkey_df_list
    gc.collect()
    
    # 保存
    print(f"  保存数据...")
    psth_path = os.path.join(output_dir, f"psth_{monkey}.npy")
    df_path = os.path.join(output_dir, f"processed_{monkey}.csv")
    
    np.save(psth_path, monkey_psth)
    monkey_df.to_csv(df_path, index=False)
    
    mem_usage = monkey_psth.nbytes / 1024**2
    print(f"  已保存: {psth_path}")
    print(f"  已保存: {df_path}")
    print(f"  形状: PSTH {monkey_psth.shape}, DataFrame {monkey_df.shape}")
    print(f"  实际内存占用: {mem_usage:.1f} MB")
    
    # 完全释放该猴子的内存
    del monkey_psth, monkey_df
    gc.collect()
    print(f"  内存已释放")

print("\n" + "="*50)
print("所有猴子数据处理完成！")
print("="*50)

逐只猴子处理数据（内存优化版）...

处理猴子: JianJian
------------------------------
  找到 35 个sessions


TypeError: 'memmap' object does not support the context manager protocol

In [ ]:
a = np.load("/media/ubuntu/sda/TrippleN/training_output/response_matrix_train.npy")

In [ ]:
a.shape

(800, 2711, 35)

In [ ]:
psth = np.load("/media/ubuntu/sda/TrippleN/GoodUnit_by_monkey/psth_JianJian.npy")
processed = pd.read_csv("/media/ubuntu/sda/TrippleN/GoodUnit_by_monkey/processed_JianJian.csv")

In [25]:
psth.shape

(17368, 1072, 450)

In [ ]:
psth.mean(axis = 1).max()

266.60257

In [ ]:
processed_raw = loadmat("/media/ubuntu/sda/TrippleN/Processed/Processed_ses01_240629_M1_2.mat")
